# Stage 00 — Data Foundation

This notebook is the **GitHub-readable walkthrough** of Stage 00. It does not do heavy compute — it loads saved intermediates and runs the validation arms (which are cheap given the parquet on disk). The heavy lifting (CRSP daily 252-day regressions, SUV AR(3) fits, the completeness filter) happens in the CLI:

```bash
python -m data_reconstruction.pipeline --config configs/stage00_data_reconstruction.yaml          # first time, builds everything
python -m data_reconstruction.pipeline --config configs/stage00_data_reconstruction.yaml --skip-build   # subsequent runs
python -m data_reconstruction.pipeline --config configs/stage00_data_reconstruction.yaml --pull-kf      # one-time Ken French extended pull
```

**Pipeline outline**

| Section | Purpose | Reference |
|---|---|---|
| 0 | Setup, config, artifact check | — |
| 1 | Raw inputs from WRDS | Doc 1 §"Raw Data Sources" |
| 2 | CRSP cleaning + Shumway delisting | Doc 1 §"Excess Returns and the Delisting Adjustment" |
| 3 | Accounting characteristics (PIT merge) | Doc 1 §"PIT mechanics" + Doc 2 Value/Profitability/Investment/Other |
| 4 | Monthly characteristics (momentum, NI, Rel2High, SUV) | Doc 2 Momentum + Liquidity |
| 5 | Risk characteristics (252d CAPM, Roll spread) | Doc 2 Risk + Doc 1 §"Roll (1984) Implied Bid-Ask Spread" |
| 6 | Final panel assembly + splits | Doc 1 §"Rank Normalization" + §"Completeness Filter" |
| 7 | Arm 3 — internal audits (gate) | Doc 3 §"Arm 3" |
| 8 | Arm 1 — CPZ distributional comparison (diagnostic) | Doc 3 §"Arm 1" |
| 9 | Arm 2 — FF5+UMD vs Ken French (gate) | Doc 3 §"Arm 2" |
| 10 | Stage 00 acceptance summary | Doc 3 §"Validation Failure Protocol" |

## 0. Setup and config

Load the package config and a small set of utility imports. The package source lives in `src/data_reconstruction/`; the notebook only imports from it.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_reconstruction import Stage00Config, run_stage00
from data_reconstruction.pipeline import _configure_logging

config = Stage00Config.from_yaml(PROJECT_ROOT / 'configs' / 'stage00_data_reconstruction.yaml')
_configure_logging(verbose=False)

FIGURES = config.figures_dir if config.figures_dir.is_absolute() else PROJECT_ROOT / config.figures_dir
FOUNDATION = config.foundation_dir if config.foundation_dir.is_absolute() else PROJECT_ROOT / config.foundation_dir
RAW = config.raw_dir
OUT = config.output_dir

print(f'Raw          : {RAW}')
print(f'Foundation   : {FOUNDATION}')
print(f'Figures      : {FIGURES}')
print(f'Splits (S1+) : {OUT}')

### Artifact check

The notebook expects the validation artifacts on disk. If they're missing, this cell runs `run_stage00` with `skip_build=True` (cheap — it reuses raw intermediates and only regenerates the assembled panel + validation outputs).

In [ ]:
required = [
    FOUNDATION / 'factor_panel_v2.parquet',
    FOUNDATION / 'stage00_summary.json',
    FOUNDATION / 'arm3_summary.json',
    FOUNDATION / 'arm1_summary.json',
]
missing = [p for p in required if not p.exists()]
if missing:
    print('Missing artifacts; running run_stage00(skip_build=True). This takes ~1 minute.')
    run_stage00(config, pull_raw=False, skip_build=True, run_validation=True, run_audits=True, run_arm1=True, run_arm2=True, write_figures=True)
else:
    print('All required artifacts present. Loading from disk.')

## 1. Raw inputs

All raw data lands in `config.raw_dir` (gitignored). Stage 00 reads from these files and never re-pulls unless `--pull-raw` is passed.

| File | Source | Doc 1 reference |
|---|---|---|
| `crsp_msf_raw.parquet` | CRSP monthly stock file | §"CRSP Monthly Stock File (MSF)" |
| `crsp_dsf_raw.parquet` | CRSP daily stock file | §"CRSP Daily Stock File (DSF)" |
| `crsp_delist_raw.parquet` | CRSP delisting returns | §"CRSP Delisting Returns" |
| `compustat_annual_raw.parquet` | Compustat funda + CCM link | §"Compustat Annual Fundamentals" |
| `ff_factors_monthly_full.parquet` | Fama-French monthly (RF + 3 factors) | §"Fama-French Factors" |
| `ff_factors_monthly_extended.parquet` | Ken French extended (FF5 + UMD + RF) | Arm 2 only |
| `ff_factors_daily.parquet` | Fama-French daily (RF + MKTRF) | Used by 252-day risk regressions |


In [ ]:
raw_files = sorted(RAW.glob('*.parquet'))
rows = []
for p in raw_files:
    df = pd.read_parquet(p, columns=None)
    rows.append({
        'file': p.name,
        'rows': f'{len(df):,}',
        'cols': df.shape[1],
        'size_mb': f'{p.stat().st_size / 1024**2:,.1f}',
    })
    del df
pd.DataFrame(rows)

## 2. CRSP cleaning — Shumway delisting and excess returns

Per Doc 1 §"Excess Returns and the Delisting Adjustment", we merge `dlret` into the monthly return series and impute Shumway (1997) values for missing `dlret` on performance-related delisting codes (`dlstcd` ∈ {500} ∪ [520, 584]):

$$
r_{i,t} = \begin{cases} \mathrm{dlret}_{i,t} & \text{if observed} \\ -0.30 & \text{NYSE/AMEX, performance delisting, missing dlret} \\ -0.55 & \text{NASDAQ, performance delisting, missing dlret} \\ 0 & \text{otherwise} \end{cases}
$$

Then $r^{ex}_{i,t} = r_{i,t} - r_{f,t}$ and $\mathrm{ME}_{i,t} = |\mathrm{prc}_{i,t}| \cdot \mathrm{shrout}_{i,t}$ (CRSP `shrout` is in thousands).

In [ ]:
clean = pd.read_parquet(RAW / 'crsp_clean_monthly.parquet', columns=['permno','date','ret','ret_adj','ret_excess','me','rf','dlstcd','dlret','dlret_fill'])
n_perf_dl = ((clean['dlstcd'].between(520, 584) | (clean['dlstcd'] == 500)) & clean['dlret'].isna()).sum()
print(f'CRSP clean monthly rows : {len(clean):,}')
print(f'Unique permnos          : {clean["permno"].nunique():,}')
print(f'Date range              : {clean["date"].min().date()}..{clean["date"].max().date()}')
print(f'Shumway-imputed delists : {n_perf_dl:,} (NYSE/AMEX → -0.30 ; NASDAQ → -0.55)')
print(f'ret_excess mean / std   : {clean["ret_excess"].mean():.4f} / {clean["ret_excess"].std():.4f}')
clean.head(3)

## 3. Accounting characteristics — PIT merge

Compustat funda is computed at `datadate` (fiscal year-end). We construct Davis-Fama-French book equity using the standard hierarchy (`seq` → `ceq + pstk` → `at - lt`), apply the 6-month publication lag to get `avail_date`, and merge into the monthly panel via `pd.merge_asof(..., direction='backward', tolerance=380 days)`.

Audit fix 2 (Doc 1): no `fillna(0)` on lagged NOA when computing AC. Audit fix 3 lives in Section 4 (Rel2High split adjustment).

In [ ]:
acc = pd.read_parquet(RAW / 'accounting_chars.parquet')
print(f'Accounting firm-years  : {len(acc):,}')
print(f'AC defined             : {acc["AC"].notna().sum():,} ({100*acc["AC"].notna().mean():.1f}%)  [Doc 1 audit fix 2 in effect]')
print(f'BE_raw available       : {acc["BE_raw"].notna().sum():,} ({100*acc["BE_raw"].notna().mean():.1f}%)')
print()
print('Sample row (PIT-merged input to the panel):')
acc[['permno','datadate','avail_date','BE_raw','IB_raw','ROE','Investment','AC']].head(3)

## 4. Monthly characteristics — momentum, NI, Rel2High, SUV

All eight Momentum-family characteristics plus NI, Rel2High and the LTurnover / LME liquidity proxies are computed from monthly CRSP via rolling windows on the (split-adjusted) raw return series. SUV (Standardized Unexplained Volume) is the runtime bottleneck — an AR(3) regression of log-volume on its three lags fit per firm-month over a 36-month rolling window.

Audit fix 3 (Doc 1): Rel2High uses split-adjusted prices ($|\mathrm{prc}|/\mathrm{cfacpr}$) to prevent stock splits from inflating the 12-month high.

In [ ]:
mon = pd.read_parquet(RAW / 'monthly_chars.parquet')
mon_cols = ['r12_2','r12_7','r36_13','LT_Rev','r2_1','ST_REV','NI','Rel2High','SUV','LTurnover','LME']
summary = pd.DataFrame({
    'characteristic': mon_cols,
    'non_null': [int(mon[c].notna().sum()) for c in mon_cols],
    'pct': [f'{100*mon[c].notna().mean():.1f}%' for c in mon_cols],
})
print(f'Monthly char firm-months: {len(mon):,}')
summary

## 5. Risk characteristics — 252-day CAPM and Roll spread

Per Doc 2 §"Risk Family", we run an OLS regression of daily excess returns on daily MKTRF over a 252-day rolling window ending at each month-end:

$$
r^{ex}_{i,d} = \alpha_{i,t} + \beta_{i,t} \cdot \mathrm{MKT}^{ex}_d + \epsilon_{i,d}, \quad d \in \mathcal{W}_{i,t}.
$$

Beta, MktBeta, IdioVol, Resid_Var, Variance, and the Roll (1984) implied spread are all derived from this regression (or directly from the daily return series for Spread and Variance). Minimum 60 valid daily observations required.

In [ ]:
risk = pd.read_parquet(RAW / 'risk_chars.parquet')
print(f'Risk firm-months     : {len(risk):,}')
print(f'Beta defined         : {risk["Beta"].notna().sum():,} ({100*risk["Beta"].notna().mean():.1f}%)')
print(f'Beta mean / median   : {risk["Beta"].mean():.3f} / {risk["Beta"].median():.3f}')
print(f'IdioVol mean (annualized) : {risk["IdioVol"].mean():.3f}')
print(f'Spread (Roll) mean   : {risk["Spread"].mean():.4f}  [zero when serial cov is non-negative]')

## 6. Final panel — completeness filter and rank normalization

The panel keeps only firm-months with **all 46 characteristics simultaneously non-null** (Doc 1 §"Completeness Filter"). Then each characteristic is cross-sectionally rank-normalized to the open interval $(-1/2, +1/2)$ within each month-end:

$$
z_{i,t,f} = \frac{R_{i,t,f}}{N_t + 1} - \tfrac{1}{2}.
$$

Splits (training 1972-2010, OOS 2011-2021, holdout 2022-2024) are written to `config.output_dir`; the pre-sample 1967-1971 remains in the panel for lookback.

In [ ]:
stage_summary = json.loads((FOUNDATION / 'stage00_summary.json').read_text())
print(json.dumps(stage_summary, indent=2))

In [ ]:
panel = pd.read_parquet(FOUNDATION / 'factor_panel_v2.parquet')
print(f'Final panel shape: {panel.shape}')
print()
print('Schema sample (first 12 columns):')
print(list(panel.columns[:12]))
print()
print('First 3 rows:')
panel.head(3)

In [ ]:
if (FIGURES / 'char_presence.png').exists():
    display(Markdown('**Pre-filter coverage by characteristic** — what fraction of firm-months have each characteristic non-null *before* the completeness filter drops rows missing any of the 46. Low values explain why the completeness filter removes about 36% of firm-months.'))
    display(Image(str(FIGURES / 'char_presence.png')))

## 7. Arm 3 — internal audits  (gate)

Per Doc 3 §"Arm 3", four audits run on the assembled panel:

- **PIT bounds**: every row satisfies `avail_date ≤ date ≤ avail_date + 380 days` (equivalently `datadate + 6m ≤ date ≤ datadate + 6m + 380d`). Hard gate.
- **Rank-normalization**: every characteristic value lies strictly in $(-1/2, +1/2)$. Hard gate.
- **Schema**: no duplicate `(permno, date)` rows; monotonic dates within permno; required columns present. Hard gate.
- **Coverage stability**: firm count changes smoothly year-over-year, modulo known transitions (1986 NASDAQ-Compustat integration). Hard gate if any non-known year exceeds 30% YoY.

In [ ]:
arm3 = json.loads((FOUNDATION / 'arm3_summary.json').read_text())
rows = []
for name, r in arm3['audits'].items():
    rows.append({
        'audit': name,
        'passed': r['passed'],
        'severity': r['severity'],
        'n_checked': f'{r["n_checked"]:,}',
        'n_violations': r['n_violations'],
        'message': r['message'][:80],
    })
pd.DataFrame(rows)

In [ ]:
if (FIGURES / 'panel_coverage.png').exists():
    display(Markdown('**Panel coverage over time** — N_t (firms) on the left axis, YoY change on the right axis. Jumps outside known transitions trigger Arm 3 hard fails.'))
    display(Image(str(FIGURES / 'panel_coverage.png')))

## 8. Arm 1 — CPZ distributional comparison  (diagnostic, no formal gate)

The CPZ reference panel is permno-anonymized at source (Stefan Jansen's replication archive distributes the file with `date + ret + 46 char` columns and no firm identifier). Per-(permno, date) MAD and Spearman as originally written in Doc 3 are therefore impossible. Arm 1 instead computes four distributional diagnostics:

- **D1** — Annual breadth (firms per month) comparison.
- **D2** — Annual equal-weighted excess return comparison.
- **D3** — Per-characteristic full-period stat differences.
- **D4** — Per-(year, char) pre-filter coverage of *our* panel (no CPZ analog).

These are reported as observed without pass/fail gating — the acceptance bar lives at Arms 2 and 3.

In [ ]:
arm1 = json.loads((FOUNDATION / 'arm1_summary.json').read_text())
print(json.dumps(arm1, indent=2))

In [ ]:
for label, fname in [
    ('**D1 Annual breadth (firms per month) — Ours vs CPZ**', 'arm1_annual_breadth.png'),
    ('**D2 Annual equal-weighted excess return — Ours vs CPZ**', 'arm1_annual_returns.png'),
    ('**D3 Per-characteristic full-period stat differences (rank-norm scale)**', 'arm1_char_stats.png'),
    ('**D4 Pre-filter characteristic coverage by year (our panel)**', 'arm1_yearly_coverage.png'),
]:
    p = FIGURES / fname
    if p.exists():
        display(Markdown(label))
        display(Image(str(p)))

## 9. Arm 2 — FF5+UMD vs Ken French  (gate)

On 2017-2024 (the post-CPZ window), we build the standard 2×3 NYSE-breakpoint portfolios on our panel and compare the resulting factor returns to Ken French's published series. Acceptance criteria (Doc 3 §"Arm 2"):

1. Pearson ρ > 0.85 for each of {MKT-RF, SMB, HML, RMW, CMA, UMD}.
2. MAD < 1% per month.
3. |annualized divergence| < 3%.
4. No calendar-year ρ below 0.80.

Known caveat: our panel excludes financial firms (`sic 6000-6999`); French includes them. Some divergence is expected on the margin; the magnitude is the diagnostic.

In [ ]:
arm2_path = FOUNDATION / 'arm2_summary.json'
if not arm2_path.exists():
    display(Markdown('**Arm 2 not yet run.** Run `python -m data_reconstruction.pipeline --config configs/stage00_data_reconstruction.yaml --pull-kf` first (one-time interactive WRDS pull), then re-execute the pipeline.'))
else:
    arm2 = json.loads(arm2_path.read_text())
    print(f'Arm 2 passed: {arm2["arm2_passed"]}')
    print(f'Months compared: {arm2["n_months_compared"]}')
    print()
    diag = pd.DataFrame(arm2['diagnostics'])
    diag['pearson_rho']  = diag['pearson_rho'].round(3)
    diag['mad']          = diag['mad'].round(4)
    diag['ann_ours']     = diag['ann_ours'].round(4)
    diag['ann_kf']       = diag['ann_kf'].round(4)
    diag['delta_ann']    = diag['delta_ann'].round(4)
    diag['min_year_rho'] = diag['min_year_rho'].round(3)
    display(diag)

In [ ]:
for label, fname in [
    ('**Cumulative factor returns — Ours vs Ken French**', 'arm2_cumulative_returns.png'),
    ('**Per-factor Pearson correlation with Ken French**', 'arm2_correlation_summary.png'),
]:
    p = FIGURES / fname
    if p.exists():
        display(Markdown(label))
        display(Image(str(p)))

## 10. Stage 00 acceptance summary

`stage0_acceptance.json` consolidates the hard-gate status (Arms 2 + 3) and the diagnostic context (Arm 1). Stage 1 should refuse to consume the panel unless `stage0_complete == true`.

In [ ]:
p = FOUNDATION / 'stage0_acceptance.json'
if not p.exists():
    print('stage0_acceptance.json not yet written; run the pipeline end-to-end.')
else:
    acc = json.loads(p.read_text())
    mechanical = acc.get('all_arms_mechanically_passed', False)
    complete = acc.get('stage0_complete', False)
    if complete and mechanical:
        status = 'PASS ✓  (mechanically clean)'
    elif complete:
        status = 'PASS ✓  (with accepted universe caveats — see DECISION_LOG entry 041)'
    else:
        status = 'FAIL ✗  (unaccepted failures present — Stage 1 blocked)'
    print(f'Stage 00 status: {status}')
    print()
    caveats = acc.get('accepted_caveats', {})
    if any(caveats.values()):
        print('Accepted caveats (universe-attributable; financials excluded vs KF includes them):')
        if caveats.get('arm3'):
            print(f'  Arm 3 audits: {caveats["arm3"]}')
        if caveats.get('arm2'):
            print(f'  Arm 2 factor-criteria: {caveats["arm2"]}')
        print()
    unaccepted = acc.get('unaccepted_failures', {})
    has_unaccepted = bool(unaccepted.get('arm3')) or bool(unaccepted.get('arm2'))
    if has_unaccepted:
        print('UNACCEPTED failures (block Stage 1):')
        if unaccepted.get('arm3'):
            print(f'  Arm 3: {unaccepted["arm3"]}')
        if unaccepted.get('arm2'):
            print(f'  Arm 2: {unaccepted["arm2"]}')
        print()
    print('Full acceptance record:')
    print(json.dumps(acc, indent=2))